# Benchmark ICDAS v4 — 7 clasificadores × CV 5-fold por niño
**Settings:** GPU T4 x2, **Internet ON** (pesos preentrenados). **Input:** salida de `base-icdas-v4` (`icdas_v4.zip` + `manifest_privado_v4.csv`). Opcional: salida de una corrida anterior de este notebook, para reanudar. Diferencia con v3: todos los recortes a escala nativa uniforme (corrige el atajo por resolución).

**Protocolo común** (igual para los 7):
- Datos: base ICDAS v4, 3.466 recortes de 411 niños; CV 5-fold agrupada por niño (folds del manifiesto). Ronda k: prueba = fold k, validación = fold (k+1) mod 5, entrenamiento = los otros 3.
- Checkpoint elegido por **macro-F1 de validación**; el fold de prueba se evalúa una sola vez. **Sin TTA.** Se guardan los pesos de cada modelo y fold (para Grad-CAM y el pipeline).
- Dos folds en paralelo, uno por GPU (sin DataParallel).

**Modelos:** ResNet50, ConvNeXt V2-T, ViT-S/16, Swin-T, CoAtNet-0 y MaxViT-T con la **receta propia de cada arquitectura** (notebooks BM originales, sin re-ajustar; ImageNet-1K) + **YOLO11m-cls** con la **receta por defecto de Ultralytics** (preentrenado ImageNet; 100 épocas, batch 16, optimizador `auto`, sus augmentations; entrada 224 px como los demás; sin pesos por clase). En YOLO el entrenamiento no se corta antes: se entrena las 100 épocas y se elige la época con mejor macro-F1 de validación (Ultralytics usa por defecto la exactitud top-1).

In [ ]:
!pip -q install timm albumentations ultralytics
import os
os.environ['PYTORCH_NVML_BASED_CUDA_CHECK'] = '1'   # no inicializar CUDA en el proceso principal
import multiprocessing as mp
import glob, json, time, zipfile, random, shutil, math, copy, platform
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, timm, cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (f1_score, cohen_kappa_score, recall_score, precision_score, accuracy_score,
                             balanced_accuracy_score, matthews_corrcoef, confusion_matrix, roc_auc_score, average_precision_score)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
NGPU = torch.cuda.device_count(); print(torch.__version__, timm.__version__, A.__version__, NGPU, 'GPU')

RUN_MODELS = ['resnet50', 'convnextv2_tiny', 'vit_small', 'swin_tiny', 'coatnet_0', 'maxvit_tiny', 'yolo11m_cls']
RUN_FOLDS  = [0, 1, 2, 3, 4]
SEED, WORKERS, IMG = 42, 2, 224
TIME_LIMIT_H = 11.0
CLASSES = ['0', 'A', 'B', 'C']
IN, W, TMP = '/kaggle/input', '/kaggle/working', '/kaggle/tmp'
T0 = time.time()

## Recetas por arquitectura (de los notebooks BM originales)

YOLO11m-cls: receta por defecto de Ultralytics (ver arriba).

In [ ]:
HEAVY_BOOST  = {0: 1.5, 1: 1.0, 2: 0.7, 3: 1.8}   # ResNet50 v6 / ConvNeXt V2 v11 / ViT-S v1b / Swin-T v1 — en el sampler
HYBRID_BOOST = {0: 1.0, 1: 1.3, 2: 1.0, 3: 2.0}   # CoAtNet-0 v2 / MaxViT-T — como peso de la loss
RECIPES = {
 'resnet50': dict(src='tv', ckpt='resnet50.IMAGENET1K_V2', head='mlp_bn', lr=3e-4, wd=3e-4, bs=32, epochs=50, patience=15,
                  warmup=5, min_lr=1e-6, warm_style='e', freeze=0, loss='ce', ls=0.1, mix='mixup', mixup_a=0.2, clip=2.0,
                  boost=HEAVY_BOOST, boost_in='sampler', aug='heavy_ge'),
 'convnextv2_tiny': dict(src='timm', ckpt='convnextv2_tiny.fcmae_ft_in1k', head='ln_drop', drop=0.4, lr=1e-4, wd=5e-4, bs=16,
                  epochs=60, patience=15, warmup=8, min_lr=1e-7, warm_style='e', freeze=8, unfreeze_stages=3, unfreeze='resume',
                  loss='focal', gamma=2.0, ls=0.0, mix='cutmix_mixup', cutmix_p=0.5, mixup_a=0.2, clip=None,
                  boost=HEAVY_BOOST, boost_in='sampler', aug='heavy'),
 'vit_small': dict(src='timm', ckpt='vit_small_patch16_224.augreg_in1k', head='linear', lr=1e-4, wd=0.05, bs=32, epochs=50,
                  patience=15, warmup=5, min_lr=1e-7, warm_style='e', freeze=0, loss='ce', ls=0.1, mix='mixup', mixup_a=0.2,
                  clip=1.0, boost=HEAVY_BOOST, boost_in='sampler', aug='heavy'),
 'swin_tiny': dict(src='timm', ckpt='swin_tiny_patch4_window7_224.ms_in1k', head='linear', lr=1e-4, wd=0.05, bs=32, epochs=50,
                  patience=15, warmup=5, min_lr=1e-7, warm_style='e', freeze=0, loss='ce', ls=0.1, mix='mixup', mixup_a=0.2,
                  clip=1.0, boost=HEAVY_BOOST, boost_in='sampler', aug='heavy'),
 'coatnet_0': dict(src='timm', ckpt='coatnet_0_rw_224.sw_in1k', head='ln_mlp', lr=3e-4, wd=0.05, bs=32, epochs=45, patience=12,
                  warmup=5, min_lr=0.0, warm_style='e+1', freeze=6, unfreeze_stages=2, unfreeze='restart_half',
                  loss='focal', gamma=2.0, ls=0.05, mix='cutmix_mixup', cutmix_p=0.5, mixup_a=0.3, clip=1.0,
                  boost=HYBRID_BOOST, boost_in='loss', aug='hybrid', drop_last=True),
 'maxvit_tiny': dict(src='timm', ckpt='maxvit_tiny_tf_224.in1k', head='ln_mlp', lr=3e-4, wd=0.05, bs=32, epochs=45, patience=12,
                  warmup=5, min_lr=0.0, warm_style='e+1', freeze=6, unfreeze_stages=2, unfreeze='restart_half',
                  loss='focal', gamma=2.0, ls=0.05, mix='cutmix_mixup', cutmix_p=0.5, mixup_a=0.3, clip=1.0,
                  boost=HYBRID_BOOST, boost_in='loss', aug='hybrid', drop_last=True),
 'yolo11m_cls': dict(src='ultralytics', ckpt='yolo11m-cls.pt', epochs=100, imgsz=224, batch=16, recipe='Ultralytics defaults'),
}

def aug_train(kind, mean, std):
    S = IMG
    if kind in ('heavy', 'heavy_ge'):
        t = [A.Resize(S, S), A.RandomResizedCrop(size=(S, S), scale=(0.8, 1.0)), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3),
             A.Rotate(limit=30, p=0.5), A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
             A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.4),
             A.RandomGamma(gamma_limit=(80, 120), p=0.4), A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.4), A.RandomShadow(p=0.3),
             A.OneOf([A.GaussianBlur(blur_limit=(3, 5), p=1.0), A.MedianBlur(blur_limit=5, p=1.0), A.MotionBlur(blur_limit=5, p=1.0)], p=0.3),
             A.GaussNoise(p=0.3)]
        if kind == 'heavy_ge':   # ResNet50 v6 incluía además estas dos
            t += [A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.3), A.ElasticTransform(alpha=1.0, sigma=50, p=0.2)]
        t += [A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(10, 20), hole_width_range=(10, 20), p=0.3)]
    else:   # hybrid (CoAtNet-0 v2 / MaxViT-T)
        t = [A.RandomResizedCrop(size=(S, S), scale=(0.75, 1.0), ratio=(0.9, 1.1), p=1.0), A.HorizontalFlip(p=0.5),
             A.VerticalFlip(p=0.2), A.RandomRotate90(p=0.3),
             A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
             A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
             A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=8, p=0.3),
             A.OneOf([A.GaussianBlur(blur_limit=(3, 5)), A.GaussNoise(std_range=(0.02, 0.08))], p=0.2),
             A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(0.05, 0.12), hole_width_range=(0.05, 0.12), p=0.3)]
    return A.Compose(t + [A.Normalize(mean=mean, std=std), ToTensorV2()])

def aug_eval(mean, std):
    return A.Compose([A.Resize(IMG, IMG), A.Normalize(mean=mean, std=std), ToTensorV2()])

## Datos: base ICDAS v4

In [ ]:
def find(name):
    h = glob.glob(f'{IN}/**/{name}', recursive=True); return h[0] if h else None
DATA = f'{TMP}/icdas'
if not os.path.exists(f'{DATA}/icdas_v4'): zipfile.ZipFile(find('icdas_v4.zip')).extractall(DATA)
man = pd.read_csv(f'{DATA}/icdas_v4/manifest.csv', dtype={'micdas': str})
priv = pd.read_csv(find('manifest_privado_v4.csv'))
man = man.merge(priv[['crop_id_v3', 'origen']].rename(columns={'crop_id_v3': 'crop_id'}), on='crop_id', how='left')
man['path'] = DATA + '/icdas_v4/' + man.crop_file
assert man.path.map(os.path.exists).all() and man.origen.notna().all()
man['y'] = man.micdas.map({c: i for i, c in enumerate(CLASSES)})
assert man.groupby('child_id').fold.nunique().max() == 1, 'un niño en dos folds'
print(len(man), 'recortes |', man.child_id.nunique(), 'niños'); print(pd.crosstab(man.fold, man.micdas, margins=True)); print(pd.crosstab(man.origen, man.micdas))
from concurrent.futures import ThreadPoolExecutor
_t = time.time()
with ThreadPoolExecutor(8) as ex:
    IMGS = dict(zip(man.path, ex.map(lambda p: cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB), man.path)))
print(f'{len(IMGS)} recortes en memoria en {time.time() - _t:.0f} s')

def mean_std(paths):
    s = np.zeros(3); s2 = np.zeros(3); n = 0
    for p in paths:
        x = cv2.resize(IMGS[p], (IMG, IMG)).reshape(-1, 3) / 255.0; s += x.mean(0); s2 += x.std(0); n += 1
    return (s / n).tolist(), (s2 / n).tolist()

class Crops(Dataset):
    def __init__(self, df, tf): self.p, self.y, self.tf = df.path.tolist(), df.y.tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(image=IMGS[self.p[i]])['image'], self.y[i]

## Reanudar: traer resultados de corridas anteriores

In [ ]:
_t = time.time()
for p in glob.glob(f'{IN}/**/results/*/fold*_preds.csv', recursive=True):
    m = p.split('/')[-2]; os.makedirs(f'{W}/results/{m}', exist_ok=True)
    for f in glob.glob(os.path.dirname(p) + '/*'):
        dst = f'{W}/results/{m}/' + os.path.basename(f)
        if not os.path.exists(dst): shutil.copy2(f, dst)
print(sorted(glob.glob(f'{W}/results/*/fold*_preds.csv')), f'({time.time() - _t:.0f} s)')

## Modelos y entrenamiento (dos folds en paralelo, uno por GPU)

In [ ]:
class Net(nn.Module):
    def __init__(self, r):
        super().__init__()
        if r['src'] == 'tv':
            self.backbone = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2); nf = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        else:
            self.backbone = timm.create_model(r['ckpt'], pretrained=True, num_classes=0); nf = self.backbone.num_features
        h = r['head']
        if h == 'mlp_bn':  self.head = nn.Sequential(nn.Linear(nf, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 4))
        elif h == 'ln_drop': self.head = nn.Sequential(nn.LayerNorm(nf, eps=1e-6), nn.Dropout(r['drop']), nn.Linear(nf, 4))
        elif h == 'ln_mlp':  self.head = nn.Sequential(nn.LayerNorm(nf), nn.Dropout(0.5), nn.Linear(nf, 512), nn.GELU(), nn.Dropout(0.2), nn.Linear(512, 4))
        else:                self.head = nn.Linear(nf, 4)
    def forward(self, x): return self.head(self.backbone(x))
    def freeze(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_last(self, n):
        for st in list(self.backbone.stages)[-n:]:
            for p in st.parameters(): p.requires_grad = True
        for nm in ('norm_pre', 'norm'):
            if hasattr(self.backbone, nm) and isinstance(getattr(self.backbone, nm), nn.Module):
                for p in getattr(self.backbone, nm).parameters(): p.requires_grad = True

class Focal(nn.Module):
    def __init__(self, gamma, ls, w): super().__init__(); self.g, self.ls, self.w = gamma, ls, w
    def forward(self, logits, y):
        ce = F.cross_entropy(logits, y, weight=self.w, label_smoothing=self.ls, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.g) * ce).mean()

def mixup(x, y, a):
    lam = np.random.beta(a, a); i = torch.randperm(x.size(0), device=x.device); return lam * x + (1 - lam) * x[i], y, y[i], lam
def cutmix(x, y, a=1.0):
    lam = np.random.beta(a, a); i = torch.randperm(x.size(0), device=x.device); H, Wd = x.size(2), x.size(3)
    r = math.sqrt(1 - lam); cw, ch = int(Wd * r), int(H * r); cx, cy = np.random.randint(Wd), np.random.randint(H)
    x1, x2 = np.clip(cx - cw // 2, 0, Wd), np.clip(cx + cw // 2, 0, Wd); y1, y2 = np.clip(cy - ch // 2, 0, H), np.clip(cy + ch // 2, 0, H)
    x = x.clone(); x[:, :, y1:y2, x1:x2] = x[i, :, y1:y2, x1:x2]
    return x, y, y[i], 1 - ((x2 - x1) * (y2 - y1) / (H * Wd))

def lr_at(r, e):
    """LR por época, igual que los WarmupCosineScheduler de los notebooks originales."""
    def wc(base, e, warm, total, mn, style):
        if e < warm: return base * ((e + 1) if style == 'e+1' else e) / warm
        prog = (e - warm) / max(1, total - warm); return mn + (base - mn) * 0.5 * (1 + math.cos(math.pi * prog))
    if r['freeze'] and r.get('unfreeze') == 'restart_half' and e >= r['freeze']:
        return wc(r['lr'] * 0.5, e - r['freeze'], 2, r['epochs'] - r['freeze'], r['min_lr'], r['warm_style'])
    return wc(r['lr'], e, r['warmup'], r['epochs'], r['min_lr'], r['warm_style'])

@torch.no_grad()
def predict(model, dl):
    model.eval(); P = []
    for x, _ in dl: P.append(torch.softmax(model(x.to(DEV, non_blocking=True)).float(), 1).cpu())
    return torch.cat(P).numpy()

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def run(mname, k):
    out = f'{W}/results/{mname}'; os.makedirs(out, exist_ok=True)
    if os.path.exists(f'{out}/fold{k}_preds.csv'): print('ya existe', mname, k); return
    r = RECIPES[mname]; seed_all(SEED)
    te, va = man[man.fold == k], man[man.fold == (k + 1) % 5]; tr = man[~man.fold.isin([k, (k + 1) % 5])]
    mean, std = mean_std(tr.path)
    cnt = np.bincount(tr.y, minlength=4)
    sw = [(1 / cnt[y]) * (r['boost'][y] if r['boost_in'] == 'sampler' else 1.0) for y in tr.y]
    g = torch.Generator(); g.manual_seed(SEED)
    sampler = WeightedRandomSampler(sw, len(sw), replacement=True, generator=g)
    dtr = DataLoader(Crops(tr, aug_train(r['aug'], mean, std)), batch_size=r['bs'], sampler=sampler, num_workers=WORKERS,
                     pin_memory=True, drop_last=True, worker_init_fn=lambda i: np.random.seed(SEED + i))
    dva = DataLoader(Crops(va, aug_eval(mean, std)), batch_size=64, num_workers=WORKERS)
    dte = DataLoader(Crops(te, aug_eval(mean, std)), batch_size=64, num_workers=WORKERS)
    model = Net(r).to(DEV)
    if r['freeze']: model.freeze()
    w = torch.tensor([r['boost'][i] for i in range(4)], dtype=torch.float32, device=DEV) if r['boost_in'] == 'loss' else None
    crit = nn.CrossEntropyLoss(label_smoothing=r['ls']) if r['loss'] == 'ce' else Focal(r['gamma'], r['ls'], w)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=r['lr'], weight_decay=r['wd'])
    if DEV == 'cuda': torch.cuda.reset_peak_memory_stats()
    best, best_ep, bad, hist, t_train = -1.0, 0, 0, [], time.time()
    for ep in range(r['epochs']):
        if r['freeze'] and ep == r['freeze']:
            model.unfreeze_last(r['unfreeze_stages'])
            opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=r['lr'], weight_decay=r['wd'])
        for gr in opt.param_groups: gr['lr'] = lr_at(r, ep)
        model.train(); t = time.time(); L = 0.0
        for x, y in dtr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            if r['mix'] == 'cutmix_mixup' and np.random.rand() < r['cutmix_p']: x, ya, yb, lam = cutmix(x, y)
            else: x, ya, yb, lam = mixup(x, y, r['mixup_a'])
            out_ = model(x); loss = lam * crit(out_, ya) + (1 - lam) * crit(out_, yb)
            opt.zero_grad(set_to_none=True); loss.backward()
            if r['clip']: torch.nn.utils.clip_grad_norm_(model.parameters(), r['clip'])
            opt.step(); L += loss.item()
        pv = predict(model, dva); f1 = f1_score(va.y, pv.argmax(1), average='macro')
        hist.append(dict(epoch=ep + 1, lr=opt.param_groups[0]['lr'], loss=L / len(dtr), val_macro_f1=f1, sec=time.time() - t))
        print(f'{mname} f{k} ep{ep + 1:02d} lr {opt.param_groups[0]["lr"]:.2e} loss {L / len(dtr):.3f} valF1 {f1:.3f} ({time.time() - t:.0f}s)')
        if f1 > best: best, best_ep, bad = f1, ep + 1, 0; torch.save(model.state_dict(), f'{TMP}/best_{mname}_{k}.pt')
        else:
            bad += 1
            if bad >= r['patience']: break
        if (time.time() - T0) / 3600 > TIME_LIMIT_H: print('límite de tiempo: se descarta esta ronda'); return
    t_train = time.time() - t_train
    model.load_state_dict(torch.load(f'{TMP}/best_{mname}_{k}.pt'))
    if DEV == 'cuda': torch.cuda.synchronize()
    t = time.time(); pt = predict(model, dte)
    if DEV == 'cuda': torch.cuda.synchronize()
    ms_img = 1000 * (time.time() - t) / len(te)
    df = te[['crop_id', 'child_id', 'fold', 'condition', 'origen', 'icdas', 'y']].copy()
    df[[f'p_{c}' for c in CLASSES]] = pt; df['pred'] = pt.argmax(1); df['model'] = mname
    df.to_csv(f'{out}/fold{k}_preds.csv', index=False)
    torch.save({n: v.half() if v.is_floating_point() else v for n, v in model.state_dict().items()}, f'{out}/fold{k}_weights_fp16.pt')   # para Grad-CAM y el pipeline
    pd.DataFrame(hist).to_csv(f'{out}/fold{k}_history.csv', index=False)
    meta = dict(model=mname, ckpt=r['ckpt'], fold=k, best_val_macro_f1=best, best_epoch=best_ep, epochs_run=len(hist),
                train_time_s=t_train, gpu_ms_per_img=ms_img, params=sum(p.numel() for p in model.parameters()),
                vram_peak_mb=torch.cuda.max_memory_allocated() / 2**20 if DEV == 'cuda' else None,
                mean=mean, std=std, recipe={a: (b if not isinstance(b, dict) else {str(i): v for i, v in b.items()}) for a, b in r.items()},
                env=dict(torch=torch.__version__, timm=timm.__version__, albumentations=A.__version__, python=platform.python_version(),
                         gpu=torch.cuda.get_device_name(0) if DEV == 'cuda' else None))
    json.dump(meta, open(f'{out}/fold{k}_meta.json', 'w'), indent=1)
    print('>>', mname, 'fold', k, '| best ep', best_ep, '| test macro-F1', round(f1_score(df.y, df.pred, average='macro'), 3))
    del model; torch.cuda.empty_cache()

def run_yolo(mname, k):
    from ultralytics import YOLO
    import ultralytics
    from ultralytics.utils.torch_utils import strip_optimizer
    out = f'{W}/results/{mname}'; os.makedirs(out, exist_ok=True)
    if os.path.exists(f'{out}/fold{k}_preds.csv'): print('ya existe', mname, k); return
    r = RECIPES[mname]
    te, va = man[man.fold == k], man[man.fold == (k + 1) % 5]; tr = man[~man.fold.isin([k, (k + 1) % 5])]
    D = f'{TMP}/yolo_{mname}_{k}'; shutil.rmtree(D, ignore_errors=True)
    for split, d in (('train', tr), ('val', va)):
        for c in CLASSES: os.makedirs(f'{D}/{split}/{c}', exist_ok=True)
        for p, c in zip(d.path, d.micdas): os.symlink(p, f'{D}/{split}/{c}/' + os.path.basename(p))
    best_pt = f'{TMP}/{mname}_{k}_bestf1.pt'; st = dict(best=-1.0, ep=0, hist=[])
    def on_epoch(tr_):
        v = tr_.validator
        try:
            p = torch.cat(v.pred)[:, 0].numpy(); t = torch.cat(v.targets).numpy()
            f1 = f1_score(t, p, average='macro', labels=[0, 1, 2, 3], zero_division=0)
        except Exception as e:
            print('sin predicciones de validación:', e); f1 = float('nan')
        ep = tr_.epoch + 1
        try: loss = float(tr_.tloss.mean())
        except Exception: loss = float('nan')
        st['hist'].append(dict(epoch=ep, loss=loss, val_macro_f1=f1, val_top1=float(getattr(v.metrics, 'top1', np.nan))))
        print(f'{mname} f{k} ep{ep:03d} loss {loss:.3f} valF1 {f1:.3f}', flush=True)
        if f1 > st['best']: st['best'], st['ep'] = f1, ep; shutil.copy2(tr_.last, best_pt)
    model = YOLO(r['ckpt']); model.add_callback('on_fit_epoch_end', on_epoch)
    if DEV == 'cuda': torch.cuda.reset_peak_memory_stats()
    t_train = time.time()
    model.train(data=D, epochs=r['epochs'], imgsz=r['imgsz'], batch=r['batch'], device=0 if DEV == 'cuda' else 'cpu', seed=SEED,
                deterministic=True, workers=WORKERS, project=f'{TMP}/yolo_runs', name=f'{mname}_{k}', exist_ok=True, plots=False, verbose=False)
    t_train = time.time() - t_train
    assert os.path.exists(best_pt), 'no se guardó ningún checkpoint'
    strip_optimizer(best_pt, f'{out}/fold{k}_best.pt')   # pesos fp16 sin optimizador
    mb = YOLO(f'{out}/fold{k}_best.pt'); names = {v: i for i, v in mb.names.items()}; order = [names[c] for c in CLASSES]
    t = time.time(); P = []
    for res in mb.predict(list(te.path), imgsz=r['imgsz'], device=0 if DEV == 'cuda' else 'cpu', batch=64, verbose=False, stream=True):
        P.append(res.probs.data.float().cpu().numpy()[order])
    pt = np.stack(P); ms_img = 1000 * (time.time() - t) / len(te)
    df = te[['crop_id', 'child_id', 'fold', 'condition', 'origen', 'icdas', 'y']].copy()
    df[[f'p_{c}' for c in CLASSES]] = pt; df['pred'] = pt.argmax(1); df['model'] = mname
    df.to_csv(f'{out}/fold{k}_preds.csv', index=False); pd.DataFrame(st['hist']).to_csv(f'{out}/fold{k}_history.csv', index=False)
    meta = dict(model=mname, ckpt=r['ckpt'], fold=k, best_val_macro_f1=st['best'], best_epoch=st['ep'], epochs_run=len(st['hist']),
                train_time_s=t_train, gpu_ms_per_img=ms_img, params=sum(p.numel() for p in mb.model.parameters()),
                vram_peak_mb=torch.cuda.max_memory_allocated() / 2**20 if DEV == 'cuda' else None, recipe=r, class_order=CLASSES,
                env=dict(torch=torch.__version__, ultralytics=ultralytics.__version__, python=platform.python_version(),
                         gpu=torch.cuda.get_device_name(0) if DEV == 'cuda' else None))
    json.dump(meta, open(f'{out}/fold{k}_meta.json', 'w'), indent=1)
    print('>>', mname, 'fold', k, '| best ep', st['ep'], '| test macro-F1', round(f1_score(df.y, df.pred, average='macro'), 3))
    shutil.rmtree(D, ignore_errors=True); shutil.rmtree(f'{TMP}/yolo_runs/{mname}_{k}', ignore_errors=True)

def child(mname, k, gpu):
    os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu)
    (run_yolo if RECIPES[mname]['src'] == 'ultralytics' else run)(mname, k)
jobs = [(m, k) for m in RUN_MODELS for k in RUN_FOLDS]; slots = list(range(max(1, NGPU))); running = {}
while jobs or running:
    for g, (pr, m, k) in list(running.items()):
        if not pr.is_alive(): pr.join(); print(f'[{m} fold {k} | GPU {g}] código {pr.exitcode}'); del running[g]
    free = [g for g in slots if g not in running]
    while jobs and free and (time.time() - T0) / 3600 < TIME_LIMIT_H - 0.3:
        m, k = jobs.pop(0)
        if os.path.exists(f'{W}/results/{m}/fold{k}_preds.csv'): print('ya existe', m, k); continue
        g = free.pop(0); pr = mp.get_context('fork').Process(target=child, args=(m, k, g)); pr.start(); running[g] = (pr, m, k)
        print(f'[{m} fold {k}] empieza en GPU {g}')
    if jobs and not running and (time.time() - T0) / 3600 >= TIME_LIMIT_H - 0.3: print('límite de tiempo: relanzar con esta salida como input'); break
    time.sleep(10)
print('entrenamiento', round((time.time() - T0) / 3600, 2), 'h')

## Eficiencia (una vez por modelo): parámetros, GFLOPs y latencia CPU (1 imagen, 1 hilo)

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
eff = []
for m in sorted(set(RUN_MODELS) | {p.split('/')[-2] for p in glob.glob(f'{W}/results/*/fold*_preds.csv')}):
    f = f'{W}/results/{m}/efficiency.json'
    if os.path.exists(f): eff.append(json.load(open(f))); continue
    if RECIPES[m]['src'] == 'ultralytics':
        from ultralytics import YOLO
        wts = sorted(glob.glob(f'{W}/results/{m}/fold*_best.pt')); net = YOLO(wts[0] if wts else RECIPES[m]['ckpt']).model.float().eval()
        for p in net.parameters(): p.requires_grad = False
    else: net = Net(RECIPES[m]).eval()
    x = torch.randn(1, 3, IMG, IMG)
    with FlopCounterMode(display=False) as fc: net(x)
    torch.set_num_threads(1)
    with torch.no_grad():
        for _ in range(3): net(x)
        t = time.time()
        for _ in range(20): net(x)
    torch.set_num_threads(os.cpu_count())
    e = dict(model=m, params_M=sum(p.numel() for p in net.parameters()) / 1e6, gflops=fc.get_total_flops() / 2e9,
             cpu_ms_per_img_1thread=1000 * (time.time() - t) / 20)
    json.dump(e, open(f, 'w')); eff.append(e)
print(pd.DataFrame(eff).round(2).to_string())

## Métricas
- **Primaria:** macro-F1 (4 clases Merged-ICDAS). Secundarias: balanced accuracy, exactitud, κ ponderado cuadrático, MCC; precisión/recall/F1, ROC-AUC y PR-AUC por clase; calibración (Brier, ECE).
- **Tres puntos de corte** desde las mismas predicciones (sin entrenar nada): **ICDAS ≥ 1** (caries sí/no: 0 vs A+B+C), **ICDAS ≥ 3** (0+A vs B+C, tamizaje) e **ICDAS ≥ 5** (C vs resto). En cada uno: sensibilidad, especificidad, LR+, LR−, AUROC (con la suma de probabilidades de las clases positivas) y VPP/VPN a prevalencias fijadas de 10, 30 y 50 % (Bayes), en el punto de operación argmax.
- Tablas por clase (precisión, recall, F1) para Merged-ICDAS y para el binario, en el formato de Tao et al. 2026.
- **Unidades:** recorte (diente en una foto) y **niño** (¿tiene al menos un recorte ≥ corte?; sobre los recortes etiquetados de cada niño).
- Por fold (media ± DE) y fuera de fold con IC 95 % por **bootstrap de niños** (1.000 réplicas); comparación pareada de macro-F1 contra el mejor modelo con Holm.
- Subgrupos: condición (pilot / retractor) y **origen del recorte** (base / minado; del manifiesto privado). Matriz ICDAS 0–6 × predicción por modelo.

In [ ]:
import re
def ece(P, y, bins=15):
    conf, pred = P.max(1), P.argmax(1); e = 0.0
    for lo in np.linspace(0, 1, bins + 1)[:-1]:
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e

CUTS = {'icdas1': 1, 'icdas3': 2, 'icdas5': 3}   # índice de clase mínimo positivo: A+ / B+ / C

def per_class_table(d):
    y, p = d.y.values, d.pred.values; rows = []
    pr, rc, f1 = (fn(y, p, labels=[0, 1, 2, 3], average=None, zero_division=0) for fn in (precision_score, recall_score, f1_score))
    for i, c in enumerate(CLASSES): rows.append(dict(escala='Merged-ICDAS', clase=c, n=int((y == i).sum()), precision=pr[i], recall=rc[i], f1=f1[i]))
    yb, pb = (y >= 1).astype(int), (p >= 1).astype(int)
    pr, rc, f1 = (fn(yb, pb, labels=[0, 1], average=None, zero_division=0) for fn in (precision_score, recall_score, f1_score))
    for i, c in enumerate(['sin caries', 'caries']): rows.append(dict(escala='binaria (ICDAS ≥ 1)', clase=c, n=int((yb == i).sum()), precision=pr[i], recall=rc[i], f1=f1[i]))
    return pd.DataFrame(rows)

def child_level(d, c0):
    g = d.groupby('child_id').agg(t=('y', lambda v: (v >= c0).any()), s=('pred', lambda v: (v >= c0).any()))
    return g.t.values, g.s.values

def metrics(d, full=True):
    y, p = d.y.values, d.pred.values; P = d[[f'p_{c}' for c in CLASSES]].values
    out = dict(macro_f1=f1_score(y, p, average='macro', labels=[0, 1, 2, 3], zero_division=0),
               bal_acc=balanced_accuracy_score(y, p), accuracy=accuracy_score(y, p),
               kappa_w=cohen_kappa_score(y, p, weights='quadratic'), mcc=matthews_corrcoef(y, p))
    rc = recall_score(y, p, labels=[0, 1, 2, 3], average=None, zero_division=0)
    for i, c in enumerate(CLASSES): out[f'recall_{c}'] = rc[i]
    t, s = np.isin(y, [2, 3]), np.isin(p, [2, 3])
    out.update(sens_BC=(t & s).sum() / max(1, t.sum()), spec_BC=(~t & ~s).sum() / max(1, (~t).sum()))
    for nm, c0 in CUTS.items():
        tt, ss = y >= c0, p >= c0
        out[f'sens_{nm}'] = (tt & ss).sum() / max(1, tt.sum()); out[f'spec_{nm}'] = (~tt & ~ss).sum() / max(1, (~tt).sum())
    if not full: return out
    for nm, c0 in CUTS.items():
        tt, ss = y >= c0, p >= c0; se, sp = out[f'sens_{nm}'], out[f'spec_{nm}']
        out[f'lrpos_{nm}'] = se / (1 - sp) if sp < 1 else np.inf; out[f'lrneg_{nm}'] = (1 - se) / sp if sp > 0 else np.inf
        out[f'auroc_{nm}'] = roc_auc_score(tt, P[:, c0:].sum(1)) if 0 < tt.sum() < len(tt) else np.nan
        out[f'prev_{nm}'] = tt.mean()
        for pi in (0.1, 0.3, 0.5):
            out[f'ppv_{nm}_at{int(pi * 100)}'] = se * pi / max(1e-12, se * pi + (1 - sp) * (1 - pi))
            out[f'npv_{nm}_at{int(pi * 100)}'] = sp * (1 - pi) / max(1e-12, sp * (1 - pi) + (1 - se) * pi)
    pr = precision_score(y, p, labels=[0, 1, 2, 3], average=None, zero_division=0); f1c = f1_score(y, p, labels=[0, 1, 2, 3], average=None, zero_division=0)
    for i, c in enumerate(CLASSES):
        out[f'precision_{c}'] = pr[i]; out[f'f1_{c}'] = f1c[i]
        if 0 < (y == i).sum() < len(y):
            out[f'rocauc_{c}'] = roc_auc_score(y == i, P[:, i]); out[f'prauc_{c}'] = average_precision_score(y == i, P[:, i])
    out['macro_rocauc'] = np.nanmean([out.get(f'rocauc_{c}', np.nan) for c in CLASSES])
    out.update(ppv_BC=(t & s).sum() / max(1, s.sum()), npv_BC=(~t & ~s).sum() / max(1, (~s).sum()),
               auroc_BC=roc_auc_score(t, P[:, 2] + P[:, 3]) if 0 < t.sum() < len(t) else np.nan,
               brier=np.mean(np.sum((P - np.eye(4)[y]) ** 2, 1)), ece=ece(P, y))
    return out

oof = [pd.read_csv(f) for f in sorted(glob.glob(f'{W}/results/*/fold*_preds.csv'))]
if oof:
    oof = pd.concat(oof); oof.to_csv(f'{W}/results/oof_predicciones.csv', index=False)
    per = pd.DataFrame([dict(model=m, fold=k, **metrics(d)) for (m, k), d in oof.groupby(['model', 'fold'])])
    per.to_csv(f'{W}/results/metricas_por_fold.csv', index=False)
    show = ['macro_f1', 'bal_acc', 'kappa_w', 'recall_C', 'sens_BC', 'spec_BC', 'macro_rocauc', 'ece']
    print('Por fold (media ± DE):'); print(per.groupby('model')[show].agg(['mean', 'std']).round(3).to_string())
    done = [m for m, d in oof.groupby('model') if d.fold.nunique() == 5]
    print('Modelos con los 5 folds:', done)
    rng = np.random.default_rng(0); kids = np.array(sorted(oof.child_id.unique())); B = 1000
    idx = {m: {c: x for c, x in oof[oof.model == m].groupby('child_id')} for m in done}
    draws = [rng.choice(kids, len(kids)) for _ in range(B)]
    rows, boots = [], {}
    for m in done:
        pt = metrics(oof[oof.model == m]); bs = pd.DataFrame([metrics(pd.concat([idx[m][c] for c in dr]), full=False) for dr in draws]); boots[m] = bs
        rows.append({'model': m, **pt, **{f'{c}_lo': bs[c].quantile(.025) for c in bs}, **{f'{c}_hi': bs[c].quantile(.975) for c in bs}})
        print(m, 'matriz de confusión (filas = real 0/A/B/C):', confusion_matrix(oof[oof.model == m].y, oof[oof.model == m].pred, labels=[0, 1, 2, 3]).tolist())
    if rows:
        res = pd.DataFrame(rows).sort_values('macro_f1', ascending=False); res.to_csv(f'{W}/results/metricas_oof_ic95.csv', index=False)
        print(res[['model'] + [f'{c}{s}' for c in ['macro_f1', 'kappa_w', 'recall_C', 'sens_BC', 'spec_BC'] for s in ('', '_lo', '_hi')]].round(3).to_string())
        if len(done) > 1:
            top = res.model.iloc[0]; cmp = []
            for m in done:
                if m == top: continue
                dlt = boots[top].macro_f1.values - boots[m].macro_f1.values
                cmp.append(dict(vs=m, diff=res.set_index('model').macro_f1[top] - res.set_index('model').macro_f1[m],
                                lo=np.quantile(dlt, .025), hi=np.quantile(dlt, .975), p=min(1, 2 * min((dlt <= 0).mean(), (dlt >= 0).mean()))))
            cmp = pd.DataFrame(cmp).sort_values('p'); n = len(cmp)
            cmp['p_holm'] = np.minimum(1, np.maximum.accumulate([(n - i) * p for i, p in enumerate(cmp.p)]))
            print(f'Diferencia de macro-F1 de {top} contra cada modelo (bootstrap pareado por niño):'); print(cmp.round(3).to_string())
            cmp.to_csv(f'{W}/results/comparacion_pareada.csv', index=False)
    sub = pd.concat([pd.DataFrame([dict(model=m, grupo=g, valor=c, n=len(d), **metrics(d, full=False)) for (m, c), d in oof.groupby(['model', g])])
                     for g in ('condition', 'origen')])
    sub.to_csv(f'{W}/results/metricas_subgrupos.csv', index=False); print(sub[['model', 'grupo', 'valor', 'n', 'macro_f1', 'recall_B', 'recall_C', 'sens_BC']].round(3).to_string())
    pc = pd.concat([per_class_table(d).assign(model=m) for m, d in oof.groupby('model')])
    pc.to_csv(f'{W}/results/tabla_por_clase.csv', index=False); print(pc.pivot_table(index=['escala', 'clase'], columns='model', values='f1').round(3).to_string())
    cut_cols = [f'{a}_{nm}' for nm in CUTS for a in ('sens', 'spec', 'lrpos', 'lrneg', 'auroc')] + [f'{a}_{nm}_at{q}' for nm in CUTS for a in ('ppv', 'npv') for q in (10, 30, 50)]
    if rows:
        cuts = res[['model'] + cut_cols + [c for c in res.columns if re.match(r'(sens|spec)_icdas\d_(lo|hi)$', c)]]
        cuts.to_csv(f'{W}/results/metricas_cortes.csv', index=False); print(cuts[['model'] + [f'{a}_{nm}' for nm in CUTS for a in ('sens', 'spec', 'auroc')]].round(3).to_string())
    rng2 = np.random.default_rng(1); ch = []
    for m, d in oof.groupby('model'):
        for nm, c0 in (('icdas1', 1), ('icdas3', 2)):
            t, s = child_level(d, c0); n = len(t)
            se = lambda t, s: (t & s).sum() / max(1, t.sum()); sp = lambda t, s: (~t & ~s).sum() / max(1, (~t).sum())
            bs = [(se(t[i], s[i]), sp(t[i], s[i])) for i in (rng2.integers(0, n, n) for _ in range(1000))]
            ch.append(dict(model=m, corte=nm, ninos=n, con_lesion=int(t.sum()), sens=se(t, s), sens_lo=np.quantile([b[0] for b in bs], .025), sens_hi=np.quantile([b[0] for b in bs], .975),
                           spec=sp(t, s), spec_lo=np.quantile([b[1] for b in bs], .025), spec_hi=np.quantile([b[1] for b in bs], .975)))
    ch = pd.DataFrame(ch); ch.to_csv(f'{W}/results/metricas_por_nino.csv', index=False); print('Por niño:'); print(ch.round(3).to_string())
    for m, d in oof.groupby('model'):
        pd.crosstab(d.icdas.values, d.pred.map(dict(enumerate(CLASSES))).values, rownames=['ICDAS'], colnames=['pred']).to_csv(f'{W}/results/matriz_icdas_{m}.csv')
    err = oof.groupby(['model', 'icdas'])[['pred']].apply(lambda d: pd.Series({'n': len(d), **{f'pred_{c}': (d.pred == i).mean() for i, c in enumerate(CLASSES)}}))
    err.to_csv(f'{W}/results/errores_por_icdas.csv'); print(err.round(2).to_string())
print('tiempo total', round((time.time() - T0) / 3600, 2), 'h')